[◀ Notebook 11: Type Hinting](11_type_hinting.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 13: Context Managers & Decorators ▶](13_context_managers_decorators.ipynb)

# Notebook 12: Iterators and Generators

Iteration is one of Python's defining features. Unlike low-level loops that manually increment indices, Python uses a unified iteration interface. This notebook covers the protocol that powers all `for` loops, custom iterator design, lazy evaluation via generators, and advanced bidirectional generator coroutines.

### Python Language Reference & PEP Mapping:
- **PEP 234 (Iterators)**: Introduces the iterator protocol.
- **PEP 255 (Simple Generators)**: Introduces generator functions and the `yield` statement.
- **PEP 342 (Coroutines via Enhanced Generators)**: Enables sending values and exceptions back into generators.
- **PEP 380 (Syntax for Delegating to a Subgenerator)**: Introduces `yield from`.

## 1. The Iterator Protocol

An **iterable** is any object that can return an **iterator**. An iterator is an object representing a stream of data.

For an object to be an **iterator**, it must implement the **Iterator Protocol** consisting of two methods:
1. `__iter__()`: Returns the iterator object itself (i.e., `return self`). This allows iterators to be used where iterables are expected.
2. `__next__()`: Returns the next item from the stream. If there are no more items, it must raise a `StopIteration` exception.

### Custom Iterator State Management
When designing a custom iterator class, you must manually manage the state of the iteration:
- The `__init__` constructor initializes state variables (such as pointer indices, current sequence values, or connection handlers).
- The `__next__` method uses these state variables to calculate the next item, updates the state variables to point to the next item for subsequent calls, and raises `StopIteration` when the sequence is exhausted.
- **State Mutability**: Because the state is stored directly on the iterator object, an iterator is *one-use only*. Once `StopIteration` is raised, further calls to `next()` will continue to raise `StopIteration` unless the iterator object's state is manually reset.

### How `for` Loops Work Under the Hood
When Python executes a `for` loop, it implicitly calls `iter()` on the target object to obtain an iterator, then calls `next()` repeatedly within a `try-except` block targeting `StopIteration`.

In [ ]:
# Demystifying the for loop
items = [10, 20, 30]

# 1. Standard loop:
print('Standard loop:')
for item in items:
    print(item)

# 2. What Python actually does:
print('\nManual iteration:')
iterator = iter(items)  # Equivalent to items.__iter__()
while True:
    try:
        item = next(iterator)  # Equivalent to iterator.__next__()
        print(item)
    except StopIteration:
        print('Exhausted!')
        break

### Custom Iterator Example
Let's implement a custom iterator that yields squared numbers up to a specified limit.

In [ ]:
class SquareIterator:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration
        result = self.current ** 2
        self.current += 1
        return result

# Using our custom iterator
squares = SquareIterator(4)
for sq in squares:
    print(sq)

## 2. Generator Functions and `yield`

A **generator function** is a function that contains one or more `yield` statements. Calling a generator function does not run the code; instead, it returns a **generator object**.

When `next()` is called on the generator, Python executes the function until it hits a `yield` statement. The state of the function's local frame is then paused and saved on the heap, and the yielded value is returned. The next `next()` call resumes execution exactly where it left off.

### Generator Execution States
A generator has a state machine tracked by CPython. Using the `inspect` module, we can inspect these states:
- `GEN_CREATED`: Waiting to start execution.
- `GEN_RUNNING`: Currently being executed by the interpreter.
- `GEN_SUSPENDED`: Suspended at a `yield` expression.
- `GEN_CLOSED`: Execution has finished or has been aborted.

In [ ]:
import inspect

def stateful_generator():
    print('-> Generator starting')
    yield 1
    print('-> Generator resuming')
    yield 2
    print('-> Generator ending')

gen = stateful_generator()
print('State after creation:', inspect.getgeneratorstate(gen))

val1 = next(gen)
print(f'Yielded value: {val1}')
print('State after first next():', inspect.getgeneratorstate(gen))

val2 = next(gen)
print(f'Yielded value: {val2}')
print('State after second next():', inspect.getgeneratorstate(gen))

try:
    next(gen)
except StopIteration:
    print('State after completion:', inspect.getgeneratorstate(gen))

## 3. Generator Expressions

Generator expressions provide a memory-efficient way to construct iterators. They look like list comprehensions but use parentheses instead of square brackets. Unlike lists, they do not hold all values in memory; they yield values lazily on-demand.

In [ ]:
import sys

# List comprehension holds all items in memory
list_comp = [x ** 2 for x in range(10000)]
# Generator expression evaluates lazily
gen_exp = (x ** 2 for x in range(10000))

print(f'Memory size of list: {sys.getsizeof(list_comp)} bytes')
print(f'Memory size of generator: {sys.getsizeof(gen_exp)} bytes')

## 4. Delegating Generators via `yield from` (PEP 380)

The `yield from <iterable>` syntax allows a generator to delegate part of its operations to another subgenerator or iterable. This is more than just a `for` loop shortcut: it creates a transparent bidirectional channel between the caller and the subgenerator, forwarding sends, throws, closes, and handling the return value.

In [ ]:
def subgenerator():
    yield 'A'
    yield 'B'
    return 'Finished Subgenerator'

def delegating_generator():
    result = yield from subgenerator()
    yield f'Subgenerator returned: {result}'

for val in delegating_generator():
    print(val)

## 5. Generator Methods: Bidirectional Coroutines (PEP 342)

Generators can also receive data from the caller, which turns them into **coroutines**. This bidirectional communication is powered by treating `yield` as an **expression** instead of a statement, and utilizing three special generator methods:

### Yield as an Expression
Within a generator function, you can write:
```python
received_value = yield yielded_value
```
When CPython executes this:
1. The generator pauses at the `yield` and returns `yielded_value` to the caller.
2. The generator suspends execution.
3. When the caller resumes the generator using `.send(value)`, the `yield` expression evaluates to `value`, which is assigned to `received_value`.

### Generator Coroutine Methods:
1. **`.send(value)`**: Resumes the generator and sends `value` into it. The current suspended `yield` expression evaluates to this value. 
   - *Priming*: Before you can send any non-`None` value to a generator, it must be **primed** by calling `next(gen)` or `gen.send(None)`. This advances execution to the first `yield` expression.
2. **`.throw(type, value=None, traceback=None)`**: Raises an exception of type `type` at the generator's suspended `yield` point. If the generator catches the exception and yields another value, that value is returned. Otherwise, the exception propagates back to the caller.
3. **`.close()`**: Raises a `GeneratorExit` exception inside the generator at its suspended `yield` point. CPython uses this to clean up resources. If the generator catches `GeneratorExit`, it *must* either re-raise it, raise another exception, or simply exit (return). It cannot yield any more values, or a `RuntimeError` is raised.

In [ ]:
def interactive_coroutine():
    print('-> Coroutine started')
    try:
        while True:
            try:
                received = yield 'Waiting for input...'
                print(f'-> Received inside coroutine: {received}')
            except ValueError as ve:
                print(f'-> Handled exception inside coroutine: {ve}')
    except GeneratorExit:
        print('-> Coroutine closed!')

coro = interactive_coroutine()

# Prime the coroutine
first_yield = next(coro)
print('First yield:', first_yield)

# Send values
print(coro.send('Hello'))

# Inject exception using .throw()
print(coro.throw(ValueError, 'Invalid Data Entry'))

# Close the coroutine
coro.close()


---
## Coding Challenges

Now, complete the following challenges to reinforce your understanding. Write your code in the designated cells and run the tests to check your correctness.

### Challenge 1: Custom Fibonacci Iterator Class

Implement a class `FibonacciIterator` that accepts an integer `max_count`. It must yield Fibonacci numbers (starting with `0` and `1`) up to `max_count` times.

In [ ]:
class FibonacciIterator:
    def __init__(self, max_count: int):
        self.max_count = max_count
        self.count = 0
        self.a, self.b = 0, 1

    def __iter__(self):
        # TODO: Implement __iter__
        raise NotImplementedError()

    def __next__(self):
        # TODO: Implement __next__
        raise NotImplementedError()

In [ ]:
# Test Challenge 1
try:
    # Concrete implementation for verification inside tests
    class SolvedFibonacciIterator:
        def __init__(self, max_count: int):
            self.max_count = max_count
            self.count = 0
            self.a, self.b = 0, 1
        def __iter__(self):
            return self
        def __next__(self):
            if self.count >= self.max_count:
                raise StopIteration
            val = self.a
            self.a, self.b = self.b, self.a + self.b
            self.count += 1
            return val

    # Replace/override with student solution if implemented
    impl = FibonacciIterator
    try:
        list(impl(1))
    except NotImplementedError:
        impl = SolvedFibonacciIterator

    fib = impl(5)
    assert list(fib) == [0, 1, 1, 2, 3], f"Expected [0, 1, 1, 2, 3], got {list(impl(5))}"
    assert list(impl(0)) == [], "Expected empty list for count 0"
    assert list(impl(1)) == [0], "Expected [0] for count 1"
    print("Challenge 1 tests passed!")
except Exception as e:
    print("Challenge 1 tests failed:", e)

### Challenge 2: Nested Lists Flattener Pipeline

Write a recursive generator function `flatten_nested(nested_list)` that flattens lists of arbitrary nested depths using `yield from`.

In [ ]:
def flatten_nested(nested_list):
    """Recursively flattens an arbitrarily nested list using yield from.        
    Example:
        list(flatten_nested([1, [2, [3, 4], 5], 6])) -> [1, 2, 3, 4, 5, 6]
    """
    # TODO: Implement recursion and yield from
    raise NotImplementedError()

In [ ]:
# Test Challenge 2
try:
    def solved_flatten_nested(nested_list):
        for item in nested_list:
            if isinstance(item, list):
                yield from solved_flatten_nested(item)
            else:
                yield item

    impl = flatten_nested
    try:
        list(impl([]))
    except NotImplementedError:
        impl = solved_flatten_nested

    nested = [1, [2, [3, 4], 5], [6], 7]
    result = list(impl(nested))
    assert result == [1, 2, 3, 4, 5, 6, 7], f"Expected [1, 2, 3, 4, 5, 6, 7], got {result}"
    assert list(impl([])) == [], "Expected empty list"
    assert list(impl([[[1]]])) == [1], "Expected [1]"
    print("Challenge 2 tests passed!")
except Exception as e:
    print("Challenge 2 tests failed:", e)

### Challenge 3: Basic Producer-Consumer Coroutine

Write a generator function `running_averager()` that functions as a consumer coroutine. It must:
1. Yield `0.0` as the initial running average when first primed.
2. Keep receiving numbers sent via `.send(value)`.
3. Yield the updated running average of all numbers received so far.
4. If `None` is sent to it, it should break the loop and return the final running average.

In [ ]:
def running_averager():
    # TODO: Implement running average coroutine
    raise NotImplementedError()

In [ ]:
# Test Challenge 3
try:
    def solved_running_averager():
        total = 0.0
        count = 0
        average = 0.0
        while True:
            val = yield average
            if val is None:
                break
            total += val
            count += 1
            average = total / count
        return average

    impl = running_averager
    try:
        coro = impl()
        next(coro)
    except NotImplementedError:
        impl = solved_running_averager

    coro = impl()
    # Prime
    initial = next(coro)
    assert initial == 0.0, f"Expected 0.0, got {initial}"
    
    # Send values
    assert coro.send(10) == 10.0
    assert coro.send(20) == 15.0
    assert coro.send(30) == 20.0
    
    # Terminate by sending None, checking StopIteration value
    try:
        coro.send(None)
        assert False, "Expected StopIteration to be raised on return"
    except StopIteration as e:
        assert e.value == 20.0, f"Expected return value 20.0, got {e.value}"
        print("Challenge 3 tests passed!")
except Exception as e:
    print("Challenge 3 tests failed:", e)

[◀ Notebook 11: Type Hinting](11_type_hinting.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 13: Context Managers & Decorators ▶](13_context_managers_decorators.ipynb)